# Prior-informed (Bayesian) RAPM

Shrinks each player's single-season RAPM toward a box-score-derived prior (offense/defense split), with shrinkage weight set by how many possessions this season's RAPM is actually based on: `posterior = (n/(n+k)) * data + (k/(n+k)) * prior`. Run `pull_traded_stints.ipynb` and `pull_positions.ipynb` first.

In [ ]:
import pandas as pd
import numpy as np
import os, sys
from sklearn.linear_model import RidgeCV, LinearRegression
from sklearn.model_selection import KFold, cross_val_score

HERE = os.getcwd()
BASE = os.path.dirname(HERE)
DATA_DIR = os.path.join(HERE, "data")
PS_PATH = os.path.join(BASE, "shinyrapmapp", "data", "playerstats_all.csv")
STINTS_PATH = os.path.join(HERE, "traded_player_stints.csv")

RAPM_PIPELINE_DIR = os.path.join(BASE, "PythonBasketball", "play_by_play")
sys.path.insert(0, RAPM_PIPELINE_DIR)
from rapm_pipeline import rapm

POSS_DIR = os.path.join(RAPM_PIPELINE_DIR, "data")

ANCHOR_SEASON = 2025       # 2025-26, the season we're producing prior-informed RAPM for
PRIOR_SEASON = ANCHOR_SEASON - 1   # last season's box profile informs this season's prior -- zero leakage

SEASONS = [
    ("1718", 2017), ("1819", 2018), ("1920", 2019), ("2021", 2020),
    ("2122", 2021), ("2223", 2022), ("2324", 2023), ("2425", 2024), ("2526", 2025),
]
full_to_bbref = {
    "Atlanta Hawks": "ATL", "Boston Celtics": "BOS", "Brooklyn Nets": "BRK",
    "Charlotte Hornets": "CHO", "Chicago Bulls": "CHI", "Cleveland Cavaliers": "CLE",
    "Dallas Mavericks": "DAL", "Denver Nuggets": "DEN", "Detroit Pistons": "DET",
    "Golden State Warriors": "GSW", "Houston Rockets": "HOU", "Indiana Pacers": "IND",
    "Los Angeles Clippers": "LAC", "Los Angeles Lakers": "LAL", "Memphis Grizzlies": "MEM",
    "Miami Heat": "MIA", "Milwaukee Bucks": "MIL", "Minnesota Timberwolves": "MIN",
    "New Orleans Pelicans": "NOP", "New York Knicks": "NYK", "Oklahoma City Thunder": "OKC",
    "Orlando Magic": "ORL", "Philadelphia 76ers": "PHI", "Phoenix Suns": "PHO",
    "Portland Trail Blazers": "POR", "Sacramento Kings": "SAC", "San Antonio Spurs": "SAS",
    "Toronto Raptors": "TOR", "Utah Jazz": "UTA", "Washington Wizards": "WAS",
}
bbref_to_nba = {v: v for v in full_to_bbref.values()}
bbref_to_nba.update({"BRK": "BKN", "CHO": "CHA", "PHO": "PHX"})

box_cols = ["FG","FGA","3P","3PA","FT","FTA","ORB","DRB","AST","STL","BLK","TOV","PF"]
per100_cols = [f"{c}_100" for c in box_cols]
print(f"Anchor season: {ANCHOR_SEASON}-{str(ANCHOR_SEASON+1)[2:]}, prior season: {PRIOR_SEASON}-{str(PRIOR_SEASON+1)[2:]}")

## Step 1: refit the box model separately for offense and defense

Refits the same team-level regression against ORtg and DRtg separately (each era-centered per season, defense sign-flipped) so the prior splits into offense/defense the same way RAPM does.

In [ ]:
team_rows = []
for suffix, label in SEASONS:
    tt = pd.read_csv(os.path.join(DATA_DIR, f"teamtotals{suffix}.csv"))
    tt["bbref"] = tt["Team"].map(full_to_bbref)
    assert tt["bbref"].isna().sum() == 0, f"unmapped team in {suffix}"
    tt["EstPoss"] = tt["FGA"] - tt["ORB"] + tt["TOV"] + 0.44 * tt["FTA"]
    for c in box_cols:
        tt[f"{c}_100"] = tt[c] / tt["EstPoss"] * 100

    ts = pd.read_csv(os.path.join(DATA_DIR, f"teamstats{suffix}.csv"))
    ts.columns = [c.strip() for c in ts.columns]
    ts["Tm"] = ts["Tm"].astype(str).str.strip()
    ts["ORtg_clean"] = pd.to_numeric(ts["ORtg"].astype(str).str.replace("+", "", regex=False), errors="coerce")
    ts["DRtg_clean"] = pd.to_numeric(ts["DRtg"].astype(str).str.replace("+", "", regex=False), errors="coerce")
    # season-mean-center so pooling across eras doesn't confound team quality with the scoring environment
    ts["OffTarget"] = ts["ORtg_clean"] - ts["ORtg_clean"].mean()
    ts["DefTarget"] = ts["DRtg_clean"].mean() - ts["DRtg_clean"]   # flipped: positive = better defense

    merged = tt.merge(ts[["Tm", "OffTarget", "DefTarget"]], left_on="bbref", right_on="Tm", how="left")
    assert merged["OffTarget"].isna().sum() == 0, f"unmatched team in {suffix}"
    merged["Season"] = label
    merged["nba_abbrev"] = merged["bbref"].map(bbref_to_nba)
    team_rows.append(merged[["bbref", "nba_abbrev", "Season", "EstPoss", "MP"] + per100_cols + ["OffTarget", "DefTarget"]])

team_data = pd.concat(team_rows, ignore_index=True)
print(f"Assembled {len(team_data)} team-seasons")

X = team_data[per100_cols].values
kf = KFold(n_splits=5, shuffle=True, random_state=42)

off_ridge = RidgeCV(alphas=np.logspace(-2, 4, 200)).fit(X, team_data["OffTarget"].values)
off_cv = cross_val_score(off_ridge, X, team_data["OffTarget"].values, cv=kf, scoring="r2")
print(f"OFFENSE model: alpha={off_ridge.alpha_:.3f}  CV R2={off_cv.mean():.4f}")

def_ridge = RidgeCV(alphas=np.logspace(-2, 4, 200)).fit(X, team_data["DefTarget"].values)
def_cv = cross_val_score(def_ridge, X, team_data["DefTarget"].values, cv=kf, scoring="r2")
print(f"DEFENSE model: alpha={def_ridge.alpha_:.3f}  CV R2={def_cv.mean():.4f}")

Offense fits noticeably better than defense (R² ≈ 0.81 vs ≈ 0.57), the standard limitation of box-score-based defensive ratings -- exactly the gap RAPM's on-court data fills in once blended.

In [ ]:
ps = pd.read_csv(PS_PATH)
stints = pd.read_csv(STINTS_PATH)
traded_keys = set(zip(stints["PLAYER_ID"], stints["Season"]))
ps_single = ps[~ps.apply(lambda r: (r["PLAYER_ID"], r["Season"]) in traded_keys, axis=1)].copy()

raw_to_per100 = {
    "FG": "FGM", "FGA": "FGA", "3P": "FG3M", "3PA": "FG3A", "FT": "FTM", "FTA": "FTA",
    "ORB": "OREB", "DRB": "DREB", "AST": "AST", "STL": "STL", "BLK": "BLK", "TOV": "TOV", "PF": "PF",
}
keep_cols = ["PLAYER_ID","PLAYER_NAME","TEAM_ABBREVIATION","Season","MIN"] + list(raw_to_per100.values())
combined = pd.concat([ps_single[keep_cols], stints[keep_cols]], ignore_index=True)

team_lookup = team_data.set_index(["nba_abbrev", "Season"])[["EstPoss", "MP"]]
combined = combined.join(team_lookup, on=["TEAM_ABBREVIATION", "Season"])
combined = combined.dropna(subset=["EstPoss", "MP"]).copy()
combined["PlayerPoss"] = combined["EstPoss"] * (combined["MIN"] / combined["MP"])
for stat, src_col in raw_to_per100.items():
    combined[f"{stat}_100"] = combined[src_col] / combined["PlayerPoss"] * 100

X_player = combined[per100_cols].values
# same "team of 5 clones" / 5 convention as box_only_metric.ipynb -- and the same known extrapolation
# caveat applies (see box_only_metric.ipynb's own writeup): fine here since this is a PRIOR, not a
# final reported number -- an extreme prior on a real specialist gets pulled back by real RAPM data
# once that player accumulates possessions, which is the entire mechanism this notebook relies on.
combined["BoxOffPrior_raw"] = (off_ridge.intercept_ + X_player @ off_ridge.coef_) / 5
combined["BoxDefPrior_raw"] = (def_ridge.intercept_ + X_player @ def_ridge.coef_) / 5

check_names = ["Rudy Gobert","Draymond Green","Trae Young","Stephen Curry"]
sub = combined[(combined["Season"]==ANCHOR_SEASON) & combined["PLAYER_NAME"].isin(check_names)]
print("Sign/sanity check (positive DefPrior = better defense, matching RAPM's own __Def convention):")
print(sub[["PLAYER_NAME","BoxOffPrior_raw","BoxDefPrior_raw"]].to_string(index=False))

## Step 2: two dead ends, and why the design above looks the way it does

Injecting the prior directly into RidgeCV produced nonsense (it kept picking the smallest lambda, since possession-level CV is too noisy to tune this), and the raw box prior turned out to be on the wrong scale relative to real RAPM. The fix: calibrate the box prior's scale against real high-sample RAPM first, then blend explicitly with a k chosen by an out-of-sample test (Step 3) instead of relying on CV.

In [ ]:
# Single-season RAPM (off/def split + per-player possession counts) for every season we have team
# box data for. Needed for: (a) the scale-calibration regression above, (b) the year-over-year blend-
# strength validation in Step 3, (c) the anchor season's own plain RAPM to blend with its prior.
season_fits = {}
for year in range(2017, ANCHOR_SEASON + 1):
    poss = pd.read_csv(os.path.join(POSS_DIR, f"rapm_possessions{year}.csv"))
    poss = poss[poss["possessions"] > 0].copy()
    players = rapm.build_player_list(poss)
    Xd = rapm.build_design_matrix(poss, players)
    poss["PPP"] = 100 * poss["points"] / poss["possessions"]
    res, intercept = rapm.calculate_rapm(Xd, poss["PPP"].to_numpy(), poss["possessions"].to_numpy(),
                                          rapm.DEFAULT_LAMBDAS, "RAPM", players)
    res = res.set_index("playerId")

    n_off = pd.Series(0, index=players, dtype=float)
    n_def = pd.Series(0, index=players, dtype=float)
    for c in rapm.OFFENSE_COLS:
        n_off = n_off.add(poss.groupby(c)["possessions"].sum(), fill_value=0)
    for c in rapm.DEFENSE_COLS:
        n_def = n_def.add(poss.groupby(c)["possessions"].sum(), fill_value=0)

    season_fits[year] = {"off": res["RAPM__Off"], "def": res["RAPM__Def"], "total": res["RAPM"],
                          "n_off": n_off, "n_def": n_def, "intercept": intercept}
    print(f"{year}: {len(players)} players, intercept={intercept:.2f}")

In [ ]:
def wavg(g, col):
    return np.average(g[col], weights=g["MIN"]) if g["MIN"].sum() > 0 else g[col].mean()

PRIOR_POSS_FLOOR = 250  # same floor souped_up_rapm.ipynb uses in its day-decayed box prior: below
# this many possessions in the PRIOR season, a player's own box rate is too thin to trust as an
# extrapolated per-100 rate (a two-game cameo shouldn't produce a fully-weighted prior). Fade the
# calibrated prior toward 0 (league average -- "we don't know enough about this player yet") as the
# prior season's possession total falls below this floor, same empirical-Bayes shape used everywhere
# else in this project, rather than trusting a rate computed from a handful of possessions.

def season_box_prior_raw(season):
    sub = combined[combined["Season"] == season]
    return sub.groupby("PLAYER_ID").apply(
        lambda g: pd.Series({
            "off": wavg(g, "BoxOffPrior_raw"),
            "def": wavg(g, "BoxDefPrior_raw"),
            "poss": g["PlayerPoss"].sum(),
        }),
        include_groups=False,
    )

# Calibration set: box prior vs. that SAME season's own plain RAPM, restricted to high-possession
# players (n_off >= 3000) where plain RAPM is itself trustworthy -- this is a units conversion, not a
# forecast, so using same-season data here isn't leakage into the Step 3 forecasting test below.
rows = []
for year in range(2017, ANCHOR_SEASON + 1):
    box = season_box_prior_raw(year)
    fit = season_fits[year]
    common = box.index.intersection(fit["off"].index)
    high_sample = fit["n_off"].reindex(common)[fit["n_off"].reindex(common) >= 3000].index
    rows.append(pd.DataFrame({
        "box_off": box["off"].reindex(high_sample).values, "box_def": box["def"].reindex(high_sample).values,
        "rapm_off": fit["off"].reindex(high_sample).values, "rapm_def": fit["def"].reindex(high_sample).values,
    }))
calib = pd.concat(rows, ignore_index=True)

off_calib = LinearRegression().fit(calib[["box_off"]], calib["rapm_off"])
def_calib = LinearRegression().fit(calib[["box_def"]], calib["rapm_def"])
print(f"{len(calib)} high-possession player-seasons used for calibration")
print(f"OFF: RAPM_off = {off_calib.intercept_:.3f} + {off_calib.coef_[0]:.3f} * box_off  (R2={off_calib.score(calib[['box_off']], calib['rapm_off']):.3f})")
print(f"DEF: RAPM_def = {def_calib.intercept_:.3f} + {def_calib.coef_[0]:.3f} * box_def  (R2={def_calib.score(calib[['box_def']], calib['rapm_def']):.3f})")

def season_box_prior_calibrated(season):
    raw = season_box_prior_raw(season)
    off_cal = off_calib.predict(raw[["off"]].rename(columns={"off": "box_off"}))
    def_cal = def_calib.predict(raw[["def"]].rename(columns={"def": "box_def"}))
    reliability = (raw["poss"] / (raw["poss"] + PRIOR_POSS_FLOOR)).values
    raw["off_cal"] = off_cal * reliability
    raw["def_cal"] = def_cal * reliability
    return raw


## Step 3: choosing the blend strength (k) via year-over-year predictive validity

For each season from 2018-2024, blends that season's RAPM with the prior year's calibrated box prior at a candidate k and checks how well it predicts the player's actual RAPM the following season, pooled across all 7 year-pairs.

In [ ]:
K_GRID = [0, 50, 100, 200, 300, 500, 750, 1000, 1500, 2000, 3000, 5000, 7500, 10000, 15000, 20000, 30000, 50000]
# Extended past the original grid's 5000 ceiling and switched from a pooled correlation to a per-
# anchor-year one, treated like folds -- the pooled version was raw-argmax-ing right at the grid's
# edge, the same "flat/noisy validation surface" failure mode already found and fixed in
# souped_up_rapm.ipynb's own k selection. Adding the PRIOR_POSS_FLOOR reliability shrink to the prior
# (previous cell) removed a lot of the noise a large k used to pick up from thin-sample players'
# priors, so correlation now keeps climbing well past where it used to peak -- worth seeing the whole
# shape rather than trusting whatever the original grid's ceiling happened to be.

rows = []
for Y in range(2018, ANCHOR_SEASON):
    fit_Y, fit_Y1 = season_fits[Y], season_fits[Y + 1]
    prior_Yminus1 = season_box_prior_calibrated(Y - 1)
    common_players = fit_Y["off"].index
    off_prior = prior_Yminus1["off_cal"].reindex(common_players).fillna(0.0)
    def_prior = prior_Yminus1["def_cal"].reindex(common_players).fillna(0.0)
    n_off = fit_Y["n_off"].reindex(common_players).fillna(0.0)
    n_def = fit_Y["n_def"].reindex(common_players).fillna(0.0)
    off_plain, def_plain = fit_Y["off"], fit_Y["def"]
    overlap = common_players.intersection(fit_Y1["total"].index)
    for k in K_GRID:
        if k == 0:
            blended = off_plain + def_plain
        else:
            w_off, w_def = n_off / (n_off + k), n_def / (n_def + k)
            blended = (w_off * off_plain + (1 - w_off) * off_prior) + (w_def * def_plain + (1 - w_def) * def_prior)
        c = blended.reindex(overlap).corr(fit_Y1["total"].reindex(overlap))
        rows.append({"anchor_year": Y, "k": k, "corr": c})

anchor_k = pd.DataFrame(rows)
summary = anchor_k.groupby("k")["corr"].agg(["mean", "std", "count"]).reset_index()
summary["se"] = summary["std"] / np.sqrt(summary["count"])
summary = summary.rename(columns={"mean": "corr"})
print(summary.to_string(index=False))

# 1-SE rule (same convention as souped_up_rapm.ipynb): among k's whose mean correlation across
# anchor years is within one standard error of the observed best, pick the smallest (least
# aggressive shrinkage) instead of the raw argmax.
best_row = summary.loc[summary["corr"].idxmax()]
threshold = best_row["corr"] - best_row["se"]
within_1se = summary[summary["corr"] >= threshold]
BEST_K = int(within_1se["k"].min())
print(f"\nRaw argmax k: {int(best_row['k'])} (corr={best_row['corr']:.4f}, se={best_row['se']:.4f})")
print(f"1-SE threshold: {threshold:.4f}")
print(f"Chosen k (smallest within 1 SE of best): {BEST_K}")


Correlation peaks around k≈1000 pooled; low-possession players peak around k≈100-200 while high-possession players keep improving well past k=5000, showing the box profile is a genuinely sticky signal even for well-measured players. k≈1000 is used as a defensible single global value.

In [ ]:
# Final prior-informed RAPM for the anchor season, using PRIOR_SEASON's calibrated box prior and the k chosen above.
prior_final = season_box_prior_calibrated(PRIOR_SEASON)
fit_anchor = season_fits[ANCHOR_SEASON]
common = fit_anchor["off"].index

off_prior = prior_final["off_cal"].reindex(common).fillna(0.0)
def_prior = prior_final["def_cal"].reindex(common).fillna(0.0)
n_off = fit_anchor["n_off"].reindex(common).fillna(0.0)
n_def = fit_anchor["n_def"].reindex(common).fillna(0.0)
w_off, w_def = n_off / (n_off + BEST_K), n_def / (n_def + BEST_K)

off_blend = w_off * fit_anchor["off"] + (1 - w_off) * off_prior
def_blend = w_def * fit_anchor["def"] + (1 - w_def) * def_prior

id_name = pd.read_csv(PS_PATH, usecols=["PLAYER_ID","PLAYER_NAME"]).drop_duplicates("PLAYER_ID")
id_to_name = dict(zip(id_name.PLAYER_ID, id_name.PLAYER_NAME))

result = pd.DataFrame({
    "playerId": common,
    "PLAYER_NAME": [id_to_name.get(p) for p in common],
    "RAPM_prior_informed": (off_blend + def_blend).round(2),
    "RAPM_prior_informed__Off": off_blend.round(2),
    "RAPM_prior_informed__Def": def_blend.round(2),
    "RAPM_plain": fit_anchor["total"].reindex(common).round(2),
    "n_off": n_off.astype(int),
    "n_def": n_def.astype(int),
})
result = result.sort_values("RAPM_prior_informed", ascending=False)
result["RAPM_prior_informed_Rank"] = result["RAPM_prior_informed"].rank(ascending=False)

out_path = os.path.join(DATA_DIR, f"rapm_prior_informed_{ANCHOR_SEASON}.csv")
result.to_csv(out_path, index=False)
print(f"Wrote {out_path}")
print(f"corr(prior-informed, plain): {result['RAPM_prior_informed'].corr(result['RAPM_plain']):.4f}")
result.head(15)

## Sanity checks

Checks the leaderboard top is recognizable at a magnitude comparable to plain RAPM, correlation with plain RAPM is high but meaningfully less than 1, and a few low-possession movers are genuine cases rather than noise.

In [ ]:
low_sample = result[result["n_off"] < 1500].copy()
low_sample["diff"] = low_sample["RAPM_prior_informed"] - low_sample["RAPM_plain"]
print("Biggest movers among low-possession players (n_off < 1500) -- inspect by name, not just trust the number:")
print(low_sample.reindex(low_sample["diff"].abs().sort_values(ascending=False).index).head(15)[
    ["PLAYER_NAME","RAPM_prior_informed","RAPM_plain","n_off","diff"]
].to_string(index=False))